# Vorbereitungen

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
from Master import Master
from Node import Node
import Model
import numpy as np
import matplotlib.pyplot as plt
from Data_manager import DataManager
import matplotlib.pyplot as plt
import numpy as np
import keras
import tensorflow as tf
import json

## Parameter

In [ ]:
FEDAVG = 0
FEDAVGM = 1
SEED = 51588
AGGREGATION = FEDAVGM

# Hyperparameter
NUM_ROUNDS = 300
NUM_NODES = 100
FRACTION = 0.1
LOCAL_EPOCHS = 5
BATCH_SIZE = 64

MOMENTUM_BETA = 0.7
MU = 0

# Non-IID-Steuerung
LABEL_ALPHA = 0.001
QUANTITY_ALPHA = 1000000

In [ ]:
# Determinismus
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [ ]:

# Funktion als Variable speichern
# unten wird dann für jede Node ein neues modell erstellt, 
# damit nicht alle Nodes auf dasselbe Modell zugreifen
build_model = Model.create_cnn_model
# Master erzeugen
master = Master(model=build_model())

## Data Visualization

In [ ]:
def visualize_split(partitions):
    """Balkendiagramm zur Darstellung des Splits mit Dirichlet"""

    num_nodes = len(partitions)
    num_classes = 10

    stats = np.zeros((num_nodes, num_classes))
    
    for i, (x, y) in enumerate(partitions):
        labels = np.argmax(y, axis=1)
        for label in labels:
            stats[i, label] += 1
            
    plt.figure(figsize=(12, 6))
    bottom = np.zeros(num_nodes)
    
    for c in range(num_classes):
        plt.bar(range(num_nodes), stats[:, c], bottom=bottom, label=f"Klasse {c}")
        bottom += stats[:, c]
        
    plt.xlabel("Node ID")
    plt.ylabel("Anzahl Bilder")
    plt.title("Datenverteilung pro Node (Dirichlet Split)")
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Not Used
def visualize_heatmap(partitions):
    """Heatmap zur Darstellung der Datenverteilung"""
    num_nodes = len(partitions)
    num_classes = 10
    stats = np.zeros((num_nodes, num_classes))
    
    for i, (x, y) in enumerate(partitions):
        labels = np.argmax(y, axis=1)
        for label in labels:
            stats[i, label] += 1

    plt.figure(figsize=(12, 6))
    im = plt.imshow(stats, cmap='Blues')
    
    plt.colorbar(im, label='Anzahl der Bilder')
    plt.xlabel("Klasse (Ziffer 0-9)")
    plt.ylabel("Node ID")
    plt.title("Heatmap der Datenverteilung")
    plt.xticks(range(num_classes))
    plt.yticks(range(num_nodes))
    
    if num_nodes <= 15:
        for i in range(num_nodes):
            for j in range(num_classes):
                plt.text(j, i, int(stats[i, j]), ha="center", va="center", 
                         color="black" if stats[i, j] < stats.max()/2 else "white")
    
    plt.tight_layout()
    plt.show()

# Datensatz

## Partitionen erstellen

In [ ]:
dm = DataManager()
partitions = dm.create_partitions(num_nodes=NUM_NODES, label_alpha=LABEL_ALPHA, quantity_alpha=QUANTITY_ALPHA, seed=SEED)

### Visualisierungen

In [ ]:

visualize_split(partitions)
#visualize_heatmap(partitions)

## Nodes erstellen

In [ ]:
def create_nodes(num):
    nodes = []
    """Erstellen aller Nodes. Jede Node erhält eine ID, 
    ein eigenes Modell (kein Zugriff aufs selbe Modell im RAM) und Daten
    """
    for n in range (num):
        new_node = Node(node_id=n, 
                    local_model=build_model(), 
                    x_local=partitions[n][0], 
                    y_local=partitions[n][1],
                    local_epochs=LOCAL_EPOCHS,
                    batch_size=BATCH_SIZE)
        nodes.append(new_node)
    return nodes

In [ ]:
nodes = create_nodes(NUM_NODES)

# Training

In [ ]:

# Ergebnis Dictionary für die Clients
result_data = {
    "meta": {
        "seed": SEED,
        "nodes": NUM_NODES,
        "fraction": FRACTION,
        "local_epochs": LOCAL_EPOCHS,
        "mu": MU,
        "momentum_beta": MOMENTUM_BETA,
        "aggregation": AGGREGATION
    },
    "federated": {
        "global_val_acc": [],
        "global_val_loss": [],
        "weight_divergence": [],
        "layer_weight_divergence": [],
        "nodes_history": {
            f"node_{i}": {"train_acc": [], "train_loss": []} for i in range(NUM_NODES)
        }
    }
}

# Ergebnis Dictionary für SGD-Baseline
sgd_result_data = {
    "meta": {
        "seed": SEED,
        "nodes": NUM_NODES,
        "fraction": FRACTION,
        "batch_size": BATCH_SIZE*(int(round(NUM_NODES*FRACTION))),
        "epochs": 200
    },
    "sgd_baseline": {
        "train_acc": [],
        "val_acc": [],
        "train_loss": [],
        "val_loss":[]
    },
}

## SGD Baseline

In [ ]:
import os
def save_model_weights(weights, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    np.save(filepath, np.array(weights, dtype=object))

In [ ]:
def calc_weight_divergence(global_weights, baseline_weights):
    """Berechnung der Gewichtsdivergenz über alle Layer kombiniert nach
    der Formel von Zhao et. al.
    """
    global_baseline_diff_norm = np.sqrt(sum(np.sum(np.square(gw - bw)) for bw, gw in zip(baseline_weights, global_weights)))

    baseline_norm = np.sqrt(sum(np.sum(np.square(bw)) for bw in baseline_weights))

    divergence = global_baseline_diff_norm / baseline_norm

    return float(divergence)

In [ ]:
def calc_layer_weight_divergence(global_weights, baseline_weights):
    """Berechnung der Gewichtsdivergenz pro Layer nach Zhao et. al. Wird
    in der Arbeit verwendet.
    """
    layer_divergences = []

    for baseline_layer, global_layer in zip(baseline_weights, global_weights):
        if baseline_layer.size == 0:
            continue
        
        baseline_global_diff_norm = np.linalg.norm(global_layer - baseline_layer)
        baseline_norm = np.linalg.norm(baseline_layer)

        divergence = baseline_global_diff_norm / baseline_norm
        
        layer_divergences.append(float(divergence))
    return layer_divergences


In [ ]:
def get_best_sgd_weights(history, weights):
    """Zurückgeben der besten Gewichte im Training der SGD-Baseline.
    Gemessen an der besten Validierungsgenauigkeit.
    Genutzt zur Berechnung der Gewichtsdivergenz.
    """
    best_epoch = np.argmax(history.history["val_accuracy"])
    best_weights = weights[best_epoch]
    print("best Epoch", best_epoch)
    return best_weights 

In [ ]:
import SGD_Baseline
def train_sgd():
    sgd_history, sgd_weights = SGD_Baseline.sgd_train(batch_size=BATCH_SIZE*(int(round(NUM_NODES*FRACTION))), epochs=200) 
    return sgd_history, sgd_weights

In [ ]:
# Speichern der SGD-Baseline Ergebnisse
sgd_weights_filepath = f"results/{SEED}/SGD.npy"
sgd_results_filepath = f"results/{SEED}/SGD.json"
filepath_best_weights = f"results/{SEED}/SGD_BEST.npy"

if not os.path.exists(sgd_results_filepath):
    sgd_history, sgd_weights = train_sgd()
    sgd_result_data["sgd_baseline"]["train_acc"] = [float(x) for x in sgd_history.history["accuracy"]]
    sgd_result_data["sgd_baseline"]["val_acc"] = [float(x) for x in sgd_history.history["val_accuracy"]]
    sgd_result_data["sgd_baseline"]["train_loss"] = [float(x) for x in sgd_history.history["loss"]]
    sgd_result_data["sgd_baseline"]["val_loss"] = [float(x) for x in sgd_history.history["val_loss"]]

    best_sgd_weights = get_best_sgd_weights(sgd_history, sgd_weights)
    save_model_weights(best_sgd_weights, filepath_best_weights)

    with open(sgd_results_filepath, 'w') as f:
        json.dump(sgd_result_data, f, indent=4)

    plt.plot(sgd_history.history['accuracy'], 'bo', label='Trainingsgenauigkeit')
    plt.plot(sgd_history.history['val_accuracy'], 'b', label='Validierungsgenauigkeit')
    plt.title('Trainings- und Validierungsgenauigkeit')
    plt.xlabel('Epochen')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.legend()

In [ ]:
# Training der Clients

# Histories der Clients
local_histories = []
# Histories des Servers (evaluate)
global_histories = []
best_sgd_weights = np.load(filepath_best_weights, allow_pickle=True).tolist()

for round_ in range(NUM_ROUNDS):
    print(f"Global round: {round_}")
    
    current_round_weights = []
    current_round_data_counts = []
    current_global_weights = master.get_global_weights()

    round_num_nodes = max(1, int(round(NUM_NODES * FRACTION)))
    nodes_selected = np.random.choice(nodes, round_num_nodes, replace=False)

    for n in nodes_selected:
        print(f"Local Round for Node ID: {n.node_id}")
        weights, data_count, history = n.local_train(global_weights=current_global_weights, mu=MU)

        current_round_weights.append(weights)
        current_round_data_counts.append(data_count) 

        result_data["federated"]["nodes_history"][f"node_{n.node_id}"]["train_acc"].append(history.history['accuracy'][-1])
        result_data["federated"]["nodes_history"][f"node_{n.node_id}"]["train_loss"].append(history.history['loss'][-1])
        
    if AGGREGATION == FEDAVG:
        print("Using FedAvg to aggregate")
        master.FedAvg(current_round_weights, current_round_data_counts)  
    elif AGGREGATION == FEDAVGM:
        print("Using FedAvgM to aggregate")
        master.FedAvgM(current_round_weights, current_round_data_counts, MOMENTUM_BETA)
    
    weight_divergence = calc_weight_divergence(master.get_global_weights(), best_sgd_weights)
    result_data["federated"]["weight_divergence"].append(weight_divergence)
    result_data["federated"]["layer_weight_divergence"].append(calc_layer_weight_divergence(master.get_global_weights(), best_sgd_weights))

    test_x, test_y = dm.get_test_data() 
    test_results = master.evaluate(test_x, test_y)  

    result_data["federated"]["global_val_acc"].append(test_results[1])
    result_data["federated"]["global_val_loss"].append(test_results[0])


In [ ]:
# Speichern der Client Ergebnisse
if not os.path.exists(f"results/{SEED}/{AGGREGATION}_Label_{LABEL_ALPHA}_Quantity_{QUANTITY_ALPHA}_MU{MU}_Beta_{MOMENTUM_BETA}/results.json"):
    os.makedirs(f"results/{SEED}/{AGGREGATION}_Label_{LABEL_ALPHA}_Quantity_{QUANTITY_ALPHA}_MU{MU}_Beta_{MOMENTUM_BETA}/")
    with open(f"results/{SEED}/{AGGREGATION}_Label_{LABEL_ALPHA}_Quantity_{QUANTITY_ALPHA}_MU{MU}_Beta_{MOMENTUM_BETA}/results.json", 'w') as f:
        json.dump(result_data, f, indent=4)

# Ergebnisse

In [ ]:
plt.plot(result_data["federated"]["global_val_acc"], label="Globale Genauigkeit")

plt.title("Globale Accuracy")
plt.xlabel("Globale Epochen")
plt.ylabel("Genauigkeit")
plt.ylim(0,1)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.plot(result_data["federated"]["global_val_loss"], label="Globaler Loss")

plt.title("Globale Accuracy")
plt.xlabel("Globale Epochen")
plt.ylabel("Genauigkeit")

plt.legend()
plt.grid(True)
plt.show()